# Location Selection with E-NAUTILUS: Part 2
_Running of E-NAUTILUS for decision making, and presentation of results_


In [1]:
import numpy as np
import pandas as pd
import polars as pl
import pickle
import folium
# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Load results from previous session

In [2]:
file_name = "data/pf_test.pkl"

output = open(file_name, 'rb')
prev_session = pickle.load(output)

raw_ref_pf = prev_session["pf"]
prob = prev_session["prob"]
sites = prev_session["sites"]
cities = prev_session["cities"]
cities_adj2sites = prev_session["cities_adj2sites"]


## Load reference front and problem 

In [3]:

output_flat = np.array(raw_ref_pf).flatten()

def process_lists(dict2conv):
    return {key: np.array(dict2conv[key]).flatten().tolist() for key in dict2conv.keys()}

# TODO include constraints in here too
output_dict = [
    output.optimal_objectives | 
    process_lists(output.optimal_variables) 
    for output in output_flat]

results = pl.DataFrame(output_dict)

results = results.unique(subset=("f_1", "f_2", "f_3", "f_4"))

results = results.with_columns([
    (-pl.col("f_1")).alias("f_1_min"),
    (pl.col("f_2")).alias("f_2_min"),
    (pl.col("f_3")).alias("f_3_min"),
    (-pl.col("f_4")).alias("f_4_min")
])

nadir_point = {
  "f_1": float(results["f_1"].min()),
  "f_2": float(results["f_2"].max()),
  "f_3": float(results["f_3"].max()),
  "f_4": float(results["f_4"].min())
}

results = results.unique()

display(results)

print(f"Nadir point: {nadir_point}")
print(f"Nadir point (problem): {prob.get_nadir_point()}")
print(f"Idedal point (problem): {prob.get_ideal_point()}")



f_1,f_2,f_3,f_4,sv,cover,_alpha,f_1_min,f_2_min,f_3_min,f_4_min
f64,f64,f64,f64,list[f64],list[f64],list[f64],f64,f64,f64,f64
182.0,16.0,3727.755,105085.0,"[1.0, 1.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.0],-182.0,16.0,3727.755,-105085.0
104.0,0.0,1608.005,105085.0,"[-0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.428571],-104.0,0.0,1608.005,-105085.0
182.0,16.0,3727.755,0.0,"[1.0, 1.0, … 1.0]","[-0.0, -0.0, … 0.0]",[308.132484],-182.0,16.0,3727.755,-0.0
161.0,7.0,2909.575,141084.0,"[0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.124474],-161.0,7.0,2909.575,-141084.0
97.0,6.0,1813.335,141084.0,"[-0.0, 1.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.472491],-97.0,6.0,1813.335,-141084.0
…,…,…,…,…,…,…,…,…,…,…
136.0,0.0,2183.495,105085.0,"[0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.0],-136.0,0.0,2183.495,-105085.0
149.0,3.0,2559.165,141084.0,"[-0.0, -0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.181319],-149.0,3.0,2559.165,-141084.0
97.0,2.0,1667.015,141084.0,"[-0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.472491],-97.0,2.0,1667.015,-141084.0


Nadir point: {'f_1': 0.0, 'f_2': 16.0, 'f_3': 3727.755, 'f_4': 0.0}
Nadir point (problem): {'f_1': 0, 'f_2': 17, 'f_3': 3838.3199999999997, 'f_4': 0}
Idedal point (problem): {'f_1': 182, 'f_2': 0, 'f_3': 0, 'f_4': 161142}


## Filter out dominated solutions

In [4]:
from pymoo.util.nds.non_dominated_sorting import find_non_dominated

pf = results.select(["f_1", "f_2", "f_3", "f_4"]).to_numpy()

pf[:,0] = -pf[:,0]
pf[:,3] = -pf[:,3]

nd_inds = find_non_dominated(pf)
display(nd_inds)

nd_df = results[nd_inds,:]
display(nd_df)
nd_df

reachable_indices = list(range(len(nd_df)))  # everything reachable from nadir
reachable_indices


array([ 0,  1,  3,  6,  7,  8, 10, 11, 12, 13, 14])

f_1,f_2,f_3,f_4,sv,cover,_alpha,f_1_min,f_2_min,f_3_min,f_4_min
f64,f64,f64,f64,list[f64],list[f64],list[f64],f64,f64,f64,f64
182.0,16.0,3727.755,105085.0,"[1.0, 1.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.0],-182.0,16.0,3727.755,-105085.0
104.0,0.0,1608.005,105085.0,"[-0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.428571],-104.0,0.0,1608.005,-105085.0
161.0,7.0,2909.575,141084.0,"[0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.124474],-161.0,7.0,2909.575,-141084.0
0.0,0.0,0.0,0.0,"[-0.0, 0.0, … -0.0]","[-0.0, 0.0, … 0.0]",[0.0],-0.0,0.0,0.0,-0.0
12.0,0.0,152.0,6171.0,"[-0.0, 0.0, … -0.0]","[1.0, 1.0, … 0.0]",[0.039601],-12.0,0.0,152.0,-6171.0
…,…,…,…,…,…,…,…,…,…,…
136.0,0.0,2183.495,105085.0,"[0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.0],-136.0,0.0,2183.495,-105085.0
149.0,3.0,2559.165,141084.0,"[-0.0, -0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.181319],-149.0,3.0,2559.165,-141084.0
97.0,2.0,1667.015,141084.0,"[-0.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.472491],-97.0,2.0,1667.015,-141084.0


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

## Helper functions

In [5]:
# Function to determine marker size based on population
def get_marker_size(population):
    return max(5, population / 1000)  # Adjust the divisor to scale marker size

def create_color_dict(cities, ev_cities, cc): 
    marker_color = {}
    for city in cities.loc[:,"city"]: 
        if city in ev_cities: 
            marker_color[city] = "orange"
        elif city in cc: 
            marker_color[city] = "yellow"
        else: 
            marker_color[city] = "grey"

    return marker_color

def select_point(results, sol_id): 

    return {
        "f_1": int(results.loc[sol_id, "Total patients served"]),
        "f_2": int(results.loc[sol_id, "# of visited sites that are under-attended"]),
        "f_3": float(results.loc[sol_id, "Total costs ($)"]),
        "f_4": float(results.loc[sol_id, "Population with access (%)"]/100.0)
        }


def clean_results(raw_results, intermediate_point=True): 
    # Transform objectives
    if intermediate_point: 
        results = pd.DataFrame(raw_results.intermediate_points)
    else:
        results = pd.DataFrame(raw_results.optimal_objectives)

    results = results.rename(columns={
                    "f_1": "Total patients served", 
                    "f_2": "# of visited sites that are under-attended", 
                    "f_3": "Total costs ($)", 
                    "f_4": "Population with access (%)"})
    results[["Total patients served"]] =  results[["Total patients served"]].astype(int)
    results[["Population with access (%)","# of visited sites that are under-attended"]] = (results[["Population with access (%)", "# of visited sites that are under-attended"]]*100.0).round(2)
    results[["Total costs ($)"]] = (results[["Total costs ($)"]]).round(2)

    results.index.name = "Solution ID"

    return results

## Calculate HV

In [6]:
from pymoo.indicators.hv import HV

ref_point = np.array(list(nadir_point.values()))
ref_point[0] = -ref_point[0]
ref_point[3] = -ref_point[3]

ind = HV(ref_point=ref_point)

int(ind(pf))

728669196230

## Run eNAUTILUS 
### Round 1
We're going to generate some solutions. They will be poor at first, but you and the computer will slowly find the best solution that fulfills your goals and preferences. 


In [7]:
# Initialize a first solution 
from desdeo.mcdm.enautilus import enautilus_step
from desdeo.mcdm.enautilus import enautilus_get_representative_solutions

current_iter = 0
selected_point = nadir_point
total_iterations = 3
display(f"Starting with point {selected_point}")

prob.get_ideal_point()


raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you prefer?")
display(results)

"Starting with point {'f_1': 0.0, 'f_2': 16.0, 'f_3': 3727.755, 'f_4': 0.0}"

number of iterations left: 3


'Which solution to do you prefer?'

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%)
Solution ID,,,,
0,0,1066.67,2485.17,0.00
1,45,1066.67,3213.00,3502833.33
2,21,1166.67,2873.32,4682766.67


### Round 2 

In [8]:
chosen_solution = 1

In [9]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")
display(raw_results)
results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)
display("Results:")



{'f_1': 45, 'f_2': 1066, 'f_3': 3213.0, 'f_4': 35028.3333}
number of iterations left: 2


ENautilusResult(current_iteration=2, iterations_left=1, intermediate_points=[{'f_1': 22.5, 'f_2': 533.0, 'f_3': 1606.5, 'f_4': 17514.16665}, {'f_1': 90.5, 'f_2': 533.0, 'f_3': 2698.2475, 'f_4': 70056.66665}, {'f_1': 55.0, 'f_2': 534.5, 'f_3': 2188.7174999999997, 'f_4': 87755.66665}], reachable_best_bounds=[{'f_1': 65.0, 'f_2': 1.0, 'f_3': 776.325, 'f_4': 140483.0}, {'f_1': 149.0, 'f_2': 0.0, 'f_3': 1608.005, 'f_4': 141084.0}, {'f_1': 136.0, 'f_2': 0.0, 'f_3': 1164.435, 'f_4': 141084.0}], reachable_worst_bounds=[{'f_1': 22.5, 'f_2': 533.0, 'f_3': 1606.5, 'f_4': 17514.16665}, {'f_1': 90.5, 'f_2': 533.0, 'f_3': 2698.2475, 'f_4': 70056.66665}, {'f_1': 55.0, 'f_2': 534.5, 'f_3': 2188.7174999999997, 'f_4': 87755.66665}], closeness_measures=[473.46416076110955, 66.66848061733465, 62.46741096793832], reachable_point_indices=[[5, 9], [1, 6, 7, 8, 10], [1, 6, 8, 9]])

'Which solution to do you find most preferable?'

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%)
Solution ID,,,,
0,22,53300.0,1606.50,1751416.66
1,90,53300.0,2698.25,7005666.66
2,55,53450.0,2188.72,8775566.66


'Results:'

### Round 3

In [10]:
chosen_solution = 1

In [11]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)




{'f_1': 90, 'f_2': 53300, 'f_3': 2698.25, 'f_4': 70056.6666}
number of iterations left: 1


'Which solution to do you find most preferable?'

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%)
Solution ID,,,,
0,0,0.0,0.00,0.0
1,136,0.0,2183.50,10508500.0
2,65,300.0,1164.44,14048300.0


## Display final result

In [12]:
final_chosen_solution = 2

In [13]:
# Get final solution 
solutions = enautilus_get_representative_solutions(prob, raw_results, nd_df) 
solution = solutions[final_chosen_solution]
results = clean_results(solution, intermediate_point=False)

display(results)
 


,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%)
Solution ID,,,,
0,65,300.0,1164.44,14048300.0


### Result postprocessing

In [14]:
# Post process result...
raw_sites = solution.optimal_variables['sv'][0].to_list()
raw_sites = [[bool(e) for e in raw_sites]]

raw_coverage = solution.optimal_variables['cover'][0].to_list()
raw_coverage = [[bool(c) for c in raw_coverage]]

sites_visited = []
for evb in raw_sites: 
    sites_visited.append("\n".join(sites.loc[evb, "site_id"].values))

cities_covered = [] 
for cc in raw_coverage: 
    cities_covered.append("\n".join(cities.loc[cc,"city"].values))

results["Sites Visited"] = sites_visited
results["Cities covered"] = cities_covered

results

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%),Sites Visited,Cities covered
Solution ID,,,,,,
0,65,300.0,1164.44,14048300.0,lima-lima-library\nbluffton-bluffton-library\n...,Ada\nAlger\nBluffton\nCairo\nColumbus Grove\nC...


### Map preprocessing

In [15]:
# Process for the map
sites_in_cities = sites.loc[raw_sites[0],:].groupby("city").agg({"site_pretty": lambda e : '<br>'.join(e)})
sites_in_cities = sites_in_cities.to_dict()['site_pretty']
sites_in_cities

# cc cities covered
cc = set(cities_covered[0].split('\n'))

site_cities = list(sites.loc[raw_sites[0], "city"])
marker_colors = create_color_dict(cities, site_cities, cc)


## Site coverage dictionary 
site2city_mat = cities_adj2sites[raw_sites[0]].astype(bool)

site2city_dict = {}
for (c,city) in enumerate(site_cities): 
    site2city_dict[city] = set(cities.loc[site2city_mat[c,],"city"]) - {city}

adjacent_sites = {}
# Record what sites are near other cities
for site_city in site2city_dict.keys(): 
    adj_cities = site2city_dict[site_city]
    from_name = cities.loc[cities.loc[:,"city"] == site_city,["city"]].values.tolist()[0][0]

    for adj_city in adj_cities: 
        to_name = cities.loc[cities.loc[:,"city"] == adj_city ,["city"]].values.tolist()[0][0]

        if to_name not in adjacent_sites.keys(): 
            adjacent_sites[to_name] = {from_name}
        else: 
            adjacent_sites[to_name] = adjacent_sites[to_name].union({from_name})



## Render map

In [16]:
# Render map

# Create a base map
m = folium.Map(location=[cities['lat'].mean(), 
                         cities['long'].mean()], 
                         zoom_start=7) 

# Draw lines between 
for site_city in site2city_dict.keys(): 
    adj_cities = site2city_dict[site_city]
    from_loc = cities.loc[cities.loc[:,"city"] == site_city,["lat", "long"]].values.tolist()

    for adj_city in adj_cities: 
        to_loc = cities.loc[cities.loc[:,"city"] == adj_city ,["lat", "long"]].values.tolist()
        folium.PolyLine(
            locations=[to_loc[0], from_loc[0]],
            color="black"
        ).add_to(m)

# Set bounds
sw = cities.loc[:,['lat', 'long']].min().values.tolist()
ne = cities.loc[:,['lat', 'long']].max().values.tolist()
m.fit_bounds([sw,ne])

# Create tool tips 
tooltips = {}
for _, row in cities.iterrows():
    city = row['city']
    tooltips[city]=f"<b>{city}</b><br><b>Population:</b> {row['pop']}"

    if city in sites_in_cities.keys():
        tooltips[city]+= "<br><b>Sites with events:</b><br>"
        tooltips[city]+= sites_in_cities[city]
    else:
        tooltips[city]+= "<br><b>No Healthwise Clinics</b>"

    if city in adjacent_sites.keys(): 
        tooltips[city]+= "<br><b>Covered by events in: </b>"
        tooltips[city]+= "<br>".join(adjacent_sites[city])


# Add cities to the map
for _, row in cities.iterrows():
    city = row['city']
    folium.CircleMarker(
        location=(row['lat'], row['long']),
        radius=get_marker_size(row['pop']),
        color="black",
        fill=True,
        fill_color=marker_colors[city],
        fill_opacity=0.6,
        tooltip=tooltips[city]
    ).add_to(m)

display(results)
display(m)

,Total patients served,# of visited sites that are under-attended,Total costs ($),Population with access (%),Sites Visited,Cities covered
Solution ID,,,,,,
0,65,300.0,1164.44,14048300.0,lima-lima-library\nbluffton-bluffton-library\n...,Ada\nAlger\nBluffton\nCairo\nColumbus Grove\nC...
